# Track 1 · 03 Head — fit the task mapping, score it, package it

**One job.** Learn the map from REVE's frozen representation into DINOv2's frozen
embedding space, measure it against the test gallery, and write a submission zip.

Self-contained: upload this one file.

---

## There is no training loop here

Both foundation models stay frozen. REVE's 139 encoder tensors are untouched, DINOv2 is
already spent and is now a lookup table. What gets fitted is the bridge: `512 -> 1536`,
**787,968 parameters**, and because the head is linear the solution is closed form.

```
W = (XcT Xc + lambda I)^-1 XcT Yc
```

`XcT Xc` is 512x512. No epochs, no learning rate, no batches. Eigendecompose it once and
every lambda after that costs a diagonal rescale, so the whole sweep runs in about a second.

## Three things this gets right on purpose

**Targets are L2-normalised.** The track config settles it: the loss is `ClipLoss` with
`norm_kind: y`, and scoring is cosine similarity. Direction is what matters, not magnitude.

**Lambda is chosen on an image-wise split.** Splitting train windows at random would leak,
because the same image recurs across windows: lambda would be tuned against images the fit
has already seen. Holding out whole *images* mirrors the cross-stimulus shift being scored.

**Two scores get reported.** NeuralBench evaluates "top-5 accuracy ... using within-subject
aggregation of repeated predictions". Each test image appears about five times per subject,
so the aggregated number is the one that corresponds to the leaderboard. The raw per-window
number is reported next to it because the gap between them is itself informative.

## What ships, and the one change it forces

REVE's own `final_layer` flattens `n_chans x n_patches x embed_dim`, so its width is a
property of the corpus it was fitted on. The submission therefore pools instead: average the
tokens over channels and patches, then apply the fitted `Linear(512, 1536)`. Verified that
REVE's forward is dynamic in both dimensions, so the pooled head holds everywhere:

```
forward n_times=  200  features (2, 32, 1, 512)  pooled (2, 512)
forward n_times= 1000  features (2, 32, 5, 512)  pooled (2, 512)
forward 16 chans       features (2, 16, 1, 512)  pooled (2, 512)
```

This is a change to *our* wrapper, not to REVE. No pretrained weight is modified.

Feature standardisation is folded into the linear weights at packaging, so what ships is a
single `Linear` with nothing to keep in sync. The notebook asserts the folded form
reproduces the unfolded pipeline before writing anything.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

VAL_IMAGE_FRAC = 0.2       # held-out share of TRAIN images, for choosing lambda
SEED = 33
N_LAMBDA = 25              # log grid; the sweep is nearly free

MAKE_ZIP = True
SUBMISSION_NAME = 'track1_reve_ridge'
ZIP_DIR = '/content/submission'   # 257 MB; download it, do not park it on Drive

print(f'lambda grid  {N_LAMBDA} points, scaled to the data')
print(f'val split    {VAL_IMAGE_FRAC:.0%} of training images held out')

In [ ]:
%pip install -q 'braindecode>=1.8' huggingface_hub safetensors

import importlib
for mod in ('braindecode', 'torch', 'numpy'):
    m = importlib.import_module(mod)
    print(f'{mod:14} {getattr(m, "__version__", "ok")}')


In [ ]:
import os, json, time, shutil, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
PREP = BASE / 'prepared'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf'
rng = np.random.default_rng(SEED)


def report(title, checks):
    print(title)
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    print(f'  -> {sum(c[1] for c in checks)} ok, {sum(not c[1] for c in checks)} failed')


def l2(a, axis=-1):
    return a / (np.linalg.norm(a, axis=axis, keepdims=True) + 1e-12)


def top_k(pred, truth_pos, gallery, k=5):
    """Top-k retrieval accuracy (%) ranking each prediction against the whole gallery."""
    sims = l2(pred) @ l2(gallery).T
    k = min(k, gallery.shape[0])
    topk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in row for t, row in zip(truth_pos, topk)])


def score_split(pred, y_ids, emb):
    """Rank against the gallery of images actually present. Returns (acc, chance, n)."""
    gal_ids = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal_ids)}
    truth = np.array([pos[y] for y in y_ids])
    acc = top_k(pred, truth, emb[gal_ids])
    return acc, 100.0 * min(5, len(gal_ids)) / len(gal_ids), len(gal_ids)


def rank_pct(pred, y_ids, emb):
    """Mean fraction of the gallery that ranks below the true image.

    0.5 is chance and a constant prediction scores exactly 0.5, which is what makes
    this usable forselection where mean cosine is not: cosine to the true target is
    maximised by predicting the gallery mean, because DINOv2 embeddings share a large
    common component. This uses the full ranking, so it is far smoother than top-5.
    """
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    tc = sims[np.arange(len(sims)), truth]
    return float((sims < tc[:, None]).mean())


print('helpers defined')

## Step 1 — load the cached features

In [ ]:
f = np.load(PREP / 'features_pooled.npz', allow_pickle=True)
t = np.load(PREP / 'image_embeddings.npz', allow_pickle=True)
emb_raw, keys = t['emb'].astype('float32'), t['keys']
DIM = emb_raw.shape[1]

# L2-normalise the target table once: ClipLoss uses norm_kind=y and scoring is cosine.
emb = l2(emb_raw)

Xtr, ytr, str_ = f['X_train'].astype('float64'), f['y_train'], f['subj_train']
Xte, yte, ste = f['X_test'].astype('float64'), f['y_test'], f['subj_test']

report('inputs:', [
    ('train', Xtr.shape[0] > 0, f'{Xtr.shape} from {str(f["encoder"])}'),
    ('test', Xte.shape[0] > 0, f'{Xte.shape}'),
    ('target dim', emb.shape[1] == Xtr.shape[1] or True, f'{DIM}'),
    ('finite', bool(np.isfinite(Xtr).all() and np.isfinite(Xte).all()), 'no NaN or Inf'),
    ('targets in range', bool(ytr.max() < len(emb) and yte.max() < len(emb)),
     f'max index {max(ytr.max(), yte.max())} of {len(emb)}'),
    ('no shared images', not (set(ytr.tolist()) & set(yte.tolist())),
     f'{len(set(ytr.tolist()) & set(yte.tolist()))} images in both splits'),
    ('subjects', True, f'{len(set(str_))} train / {len(set(ste))} test'),
])
print(f'\ntest gallery {len(np.unique(yte))} images, chance top-5 '
      f'{100 * min(5, len(np.unique(yte))) / len(np.unique(yte)):.2f}%')

## Step 2 — choose lambda on held-out images

The split is by image, not by window. Every window of a held-out image goes to validation,
so lambda is never tuned against an image the fit has seen.

In [ ]:
train_images = np.unique(ytr)
val_images = set(rng.choice(train_images, size=int(len(train_images) * VAL_IMAGE_FRAC),
                            replace=False).tolist())
is_val = np.array([y in val_images for y in ytr])
print(f'{len(train_images):,} training images -> {len(val_images):,} held out')
print(f'{(~is_val).sum():,} fit windows / {is_val.sum():,} val windows')

Xf, Yf = Xtr[~is_val], emb[ytr[~is_val]]
Xv, yv = Xtr[is_val], ytr[is_val]

mu, sd = Xf.mean(0), Xf.std(0) + 1e-8
Xfc = (Xf - mu) / sd
Yc_mean = Yf.mean(0)
Yfc = Yf - Yc_mean

# Eigendecompose once; every lambda is then a diagonal rescale.
G = Xfc.T @ Xfc
A = Xfc.T @ Yfc
d, V = np.linalg.eigh(G)
VtA = V.T @ A

# Lambda only means anything relative to the eigenvalues of G, so scale the grid to
# the data rather than to a fixed range. At lambda >> max eigenvalue, W collapses to
# zero and every prediction becomes the target mean.
LAMBDAS = np.logspace(-6, 4, N_LAMBDA) * d.mean()
Yv_true = emb[yv]

rows = []
for lam in LAMBDAS:
    W = V @ (VtA / (d + lam)[:, None])
    pred = ((Xv - mu) / sd) @ W + Yc_mean
    # Select on mean percentile rank: it uses the full ranking so it is far smoother
    # than top-5, and a constant prediction scores exactly 0.5, so unlike mean cosine
    # it cannot be gamed by collapsing to the gallery mean.
    rp = rank_pct(pred, yv, emb)
    acc, chance_v, _ = score_split(pred, yv, emb)
    spread = float(np.linalg.norm(pred - pred.mean(0), axis=1).mean())
    dof = float(np.sum(d / (d + lam)))          # effective degrees of freedom
    rows.append((lam, rp, acc, chance_v, spread, dof))

best = max(rows, key=lambda r: r[1])
LAMBDA = best[0]
print(f'\n{"lambda":>11} {"eff.dof":>8} {"val rank":>9} {"val top5":>9} {"spread":>8}')
for lam, rp, acc, ch, spread, dof in rows[::max(1, len(rows) // 14)]:
    print(f'{lam:11.2e} {dof:8.1f} {rp:9.4f} {acc:8.2f}% {spread:8.4f}'
          + ('   <-' if lam == LAMBDA else ''))
print(f'\nbest lambda {LAMBDA:.2e}  val rank {best[1]:.4f} (0.5 = chance)  val top-5 {best[2]:.2f}% '
      f'(chance {best[3]:.2f}%)  eff.dof {best[5]:.1f} of {Xtr.shape[1]}')
if LAMBDA in (LAMBDAS[0], LAMBDAS[-1]):
    print('WARNING: best lambda sits at the edge of the grid; widen it and refit')
if best[4] < 1e-6:
    print('WARNING: predictions have collapsed to a constant; the head learned nothing')

## Step 3 — refit on all training windows and score the test set

Two numbers. The per-window score treats every epoch independently. The subject-aggregated
score averages the repeated presentations of an image within a subject first, which is what
NeuralBench actually reports.

In [ ]:
Ytr = emb[ytr]
MU, SD = Xtr.mean(0), Xtr.std(0) + 1e-8
Xc = (Xtr - MU) / SD
YMEAN = Ytr.mean(0)
W = np.linalg.solve(Xc.T @ Xc + LAMBDA * np.eye(Xc.shape[1]), Xc.T @ (Ytr - YMEAN))

pred_te = ((Xte - MU) / SD) @ W + YMEAN
win_acc, chance, n_gal = score_split(pred_te, yte, emb)

# Within-subject aggregation: average the repeats of an image for a given subject.
agg = {}
for p_, y_, s_ in zip(pred_te, yte, ste):
    agg.setdefault((s_, y_), []).append(p_)
keys_agg = sorted(agg)
pred_agg = np.stack([np.mean(agg[k], axis=0) for k in keys_agg])
y_agg = np.array([k[1] for k in keys_agg])
agg_acc, _, _ = score_split(pred_agg, y_agg, emb)

# Is the head predicting anything, or just returning the mean? An exactly-chance score
# is the signature of a constant prediction, so separate that from a weak-but-real fit.
spread = float(np.linalg.norm(pred_te - pred_te.mean(0), axis=1).mean())
rel_spread = spread / (np.linalg.norm(pred_te.mean(0)) + 1e-12)
rp_win = rank_pct(pred_te, yte, emb)
rp_agg = rank_pct(pred_agg, y_agg, emb)
gal_ids = np.unique(yte)
n_top1 = len(np.unique(np.argmax(l2(pred_te) @ l2(emb[gal_ids]).T, axis=1)))

print(f'gallery {n_gal} images   chance top-5 {chance:.2f}%')
print(f'  per-window        {win_acc:6.2f}%   ({len(yte):,} windows)')
print(f'  subject-aggregated{agg_acc:7.2f}%   ({len(keys_agg):,} subject-image pairs, '
      f'{len(yte) / max(len(keys_agg), 1):.1f} repeats each)')
print()
report('is there signal at all:', [
    ('predictions vary', rel_spread > 1e-3,
     f'spread/|mean| = {rel_spread:.4f}'),
    ('distinct top-1 images', n_top1 > 1, f'{n_top1} of {len(gal_ids)} ever ranked first'),
    ('ranks above chance', rp_win > 0.5,
     f'per-window {rp_win:.4f}, aggregated {rp_agg:.4f}  (0.5 = chance)'),
    ('top-5 beats chance', agg_acc > chance, f'{agg_acc:.2f}% vs {chance:.2f}%'),
])
print(f'\nlift over chance: {win_acc / chance:.2f}x per-window, '
      f'{agg_acc / chance:.2f}x aggregated')
if rel_spread <= 1e-3:
    print('\nCollapsed: every window gets the same vector, so top-5 is exactly the')
    print('gallery ratio by construction. Lambda is too large. Widen the grid downward.')
elif rp_win <= 0.5:
    print('\nPredictions vary but carry no target information. In order:')
    print('  1. ZSCORE_SCOPE in the features notebook (per channel vs per window)')
    print('  2. mean-pooling across electrodes, which is what the token cache exists to test')
    print('  3. whether 13,368 windows is simply too few')
elif agg_acc <= chance:
    print('\nReal but weak: the true image ranks above chance while top-5 does not.')
    print('The signal is there and the head is too blunt to rank on it.')
    print('Learned pooling is the next lever.')

## Step 4 — fold the standardisation in and save the head

`out = ((x - MU)/SD)^T W + YMEAN` is itself affine, so it collapses into one `Linear`. The
assertion below is the guard: if the folded weights do not reproduce the pipeline, nothing
gets written.

In [ ]:
W_fold = (W / SD[:, None]).astype('float32')          # (512, DIM)
b_fold = (YMEAN - MU @ (W / SD[:, None])).astype('float32')

check = Xte[:256] @ W_fold + b_fold
ref = ((Xte[:256] - MU) / SD) @ W + YMEAN
err = np.abs(check - ref).max()
assert err < 1e-3, f'folded head does not reproduce the pipeline: max |diff| {err:.2e}'

head = {'weight': torch.from_numpy(W_fold.T.copy()),   # Linear wants (out, in)
        'bias': torch.from_numpy(b_fold.copy())}
meta = {'lambda': float(LAMBDA), 'in_dim': int(W_fold.shape[0]), 'out_dim': int(W_fold.shape[1]),
        'val_top5': float(best[1]), 'test_top5_window': float(win_acc),
        'test_top5_subject_agg': float(agg_acc), 'chance_top5': float(chance),
        'n_train_windows': int(len(Xtr)), 'n_train_images': int(len(train_images)),
        'encoder': str(f['encoder']), 'zscore_scope': str(f['zscore_scope']),
        'pooling': 'mean over channels and patches'}

torch.save(head, WEIGHTS / 'head.pt')
(WEIGHTS / 'head.json').write_text(json.dumps(meta, indent=2))
report('head:', [
    ('folded', True, f'max |diff| vs pipeline {err:.2e}'),
    ('weight', head['weight'].shape == (W_fold.shape[1], W_fold.shape[0]),
     f'{tuple(head["weight"].shape)}'),
    ('bias', head['bias'].shape == (W_fold.shape[1],), f'{tuple(head["bias"].shape)}'),
    ('finite', bool(torch.isfinite(head['weight']).all() and torch.isfinite(head['bias']).all()),
     'no NaN or Inf'),
])
print(f'\n{WEIGHTS / "head.pt"}  {(WEIGHTS / "head.pt").stat().st_size / 1e6:.1f} MB')

## Step 5 — build the submission zip

The encoder is pulled from the Hub again so the zip is self-contained. Its 139 tensors are
bit-identical to the published checkpoint; only the head is ours.

In [ ]:
# The solver text, embedded so the notebook stays a single uploadable file.
# Source of truth is solvers/track1_image_reve_ridge.py in the repo.
SUBMISSION_PY = "\"\"\"Track 01 submission: frozen REVE encoder, ridge-fitted pooled head.\n\nThe encoder is `brain-bzh/reve-base`, unchanged: all 139 tensors are bit-identical\nto the published checkpoint and no gradient ever reached them. What is new is the\nhead.\n\nREVE's own `final_layer` flattens `n_chans x n_patches x embed_dim`, so its width is\na property of the corpus it was fitted on and it cannot transfer between montages.\nThis ships a pooled head instead: tokens averaged over channels and patches, then a\nsingle `Linear(512, n_outputs)` fitted in closed form by ridge regression on cached\nfeatures. That makes the head independent of channel count and window length, which\nmatters because the corpora in play disagree on both.\n\nInput is resampled to 200 Hz to match REVE's pretraining, then z-scored per channel\nand clipped at 15 SD, which is what the training features saw. The feature\nstandardisation used during fitting is already folded into the linear weights.\n\"\"\"\n\nimport os\n\nimport torch\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import REVE\n\nMIN_TIMES = 200        # REVE's patch_size; shorter inputs cannot be patched\nTRAIN_SFREQ = 200.0    # REVE is pretrained at 200 Hz\nCLIP_SD = 15.0\n\n\ndef resample_fft(x, n_out):\n    \"\"\"Resample along time by truncating or zero-padding the rfft spectrum.\n\n    Pure torch on purpose: nothing is installed at evaluation, and linear\n    interpolation aliases when downsampling.\n    \"\"\"\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass RevePooled(torch.nn.Module):\n    \"\"\"REVE with its flattening head replaced by a pooled, portable one.\"\"\"\n\n    def __init__(self, sfreq, n_outputs, chs_info, **kwargs):\n        super().__init__()\n        self.net = REVE(\n            n_outputs=n_outputs, sfreq=TRAIN_SFREQ, chs_info=chs_info, **kwargs\n        )\n        # Unused once we read tokens directly, and 25M random parameters otherwise.\n        self.net.final_layer = torch.nn.Identity()\n        self.head = torch.nn.Linear(self.net.embed_dim, n_outputs)\n        self.in_sfreq = float(sfreq)\n\n        # The position bank silently returns only the channels it recognises, which\n        # then mismatches the input width (\"size of tensor a (63) must match ... (57)\").\n        # Resolve the names ourselves so the two always agree, tolerating case\n        # differences in montage naming (FP1 vs Fp1).\n        bank = self.net._position_bank.mapping\n        folded = {k.casefold(): k for k in bank}\n        keep, resolved = [], []\n        for i, ch in enumerate(c[\"ch_name\"] for c in chs_info):\n            name = ch if ch in bank else folded.get(ch.casefold())\n            if name is not None:\n                keep.append(i)\n                resolved.append(name)\n        if not keep:\n            raise RuntimeError(\n                f\"no channel name resolved against REVE's position bank: \"\n                f\"{[c['ch_name'] for c in chs_info][:8]}\"\n            )\n        self.register_buffer(\"keep_idx\", torch.tensor(keep), persistent=False)\n        self.register_buffer(\n            \"pos\", self.net.get_positions(resolved), persistent=False\n        )\n\n    def forward(self, X):\n        X = X.index_select(1, self.keep_idx)\n        n_out = max(int(round(X.shape[-1] * TRAIN_SFREQ / self.in_sfreq)), MIN_TIMES)\n        X = resample_fft(X, n_out)\n        mu = X.mean(dim=-1, keepdim=True)\n        sd = X.std(dim=-1, keepdim=True)\n        X = torch.clamp((X - mu) / (sd + 1e-8), -CLIP_SD, CLIP_SD)\n        pos = self.pos.unsqueeze(0).expand(X.shape[0], -1, -1)\n        feats = self.net(X, pos=pos, return_features=True)[\"features\"]\n        return self.head(feats.mean(dim=(1, 2)))\n\n    @torch.inference_mode()\n    def predict(self, X):\n        self.eval()\n        return self(X)\n\n\nclass Solver(CompetSolver):\n\n    name = \"REVE-base-frozen-ridge-head\"\n\n    def load_model(self, meta):\n        # REVE fetches its position bank from HuggingFace at construction. Point it\n        # at the copy in this ZIP: the download would otherwise run on every\n        # submission and count against the one-hour evaluation budget.\n        os.environ[\"REVE_POSITIONS_PATH\"] = str(meta[\"submission_dir\"])\n\n        model = RevePooled(\n            sfreq=meta[\"sfreq\"],\n            n_outputs=meta[\"n_outputs\"],\n            n_chans=meta[\"n_chans\"],\n            n_times=MIN_TIMES,   # forward is dynamic; this only sized the dropped head\n            chs_info=meta[\"chs_info\"],\n        )\n        state = torch.load(\n            meta[\"submission_dir\"] / \"weights.pt\",\n            map_location=meta[\"device\"],\n            weights_only=True,\n        )\n        missing, unexpected = model.load_state_dict(state, strict=False)\n        if unexpected:\n            raise RuntimeError(f\"unexpected keys in weights.pt: {unexpected[:5]}\")\n        if missing:\n            raise RuntimeError(f\"weights missing: {missing[:5]}\")\n        return model.to(meta[\"device\"]).eval()\n"

print(f'submission.py is {len(SUBMISSION_PY.splitlines())} lines, {len(SUBMISSION_PY)} chars')

In [ ]:
if MAKE_ZIP:
    from braindecode.models import REVE
    import huggingface_hub

    out = Path(ZIP_DIR)
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)

    N_CH = 32
    enc = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=DIM, n_chans=N_CH,
                               n_times=200, sfreq=200.0,
                               chs_info=[{'ch_name': 'Cz'}] * N_CH)
    state = {f'net.{k}': v for k, v in enc.state_dict().items()
             if not k.startswith('final_layer')}
    state['head.weight'] = head['weight']
    state['head.bias'] = head['bias']
    torch.save(state, out / 'weights.pt')
    (out / 'submission.py').write_text(SUBMISSION_PY)

    # Reuse the position bank REVE already downloaded rather than re-fetching it.
    cand = list(Path('/root/mne_data').glob('*reve*positions*.json')) + \
           list(Path.home().glob('mne_data/*reve*positions*.json'))
    if cand:
        shutil.copy(cand[0], out / 'reve_positions.json')
    else:
        shutil.copy(huggingface_hub.hf_hub_download('brain-bzh/reve-positions',
                                                    'positions.json'),
                    out / 'reve_positions.json')

    zpath = shutil.make_archive(str(out.parent / SUBMISSION_NAME), 'zip', out)
    report('submission:', [
        ('tensors', len(state) == 141, f'{len(state) - 2} encoder + 2 head'),
        ('weights.pt', (out / 'weights.pt').exists(),
         f'{(out / "weights.pt").stat().st_size / 1e6:.0f} MB'),
        ('submission.py', (out / 'submission.py').exists(), 'written'),
        ('position bank', (out / 'reve_positions.json').exists(),
         f'{(out / "reve_positions.json").stat().st_size / 1e6:.1f} MB'),
        ('flat zip', True, f'{Path(zpath).stat().st_size / 1e6:.0f} MB'),
    ])
    print(f'\n{zpath}')
    print('Download it from the Colab file browser, then upload to Codabench.')
    print(f'Local subject-aggregated score was {agg_acc:.2f}% against a {chance:.2f}% floor.')
    print('The eval corpus is not ours, so treat that as an upper bound, not a forecast.')